# **1. Text Extraction from Case PDF**

In [ ]:
!pip install pymupdf python-docx

In [ ]:
import fitz  # PyMuPDF
import re
from docx import Document
import sys
import os


In [ ]:
import os
from kaggle_secrets import UserSecretsClient


# Get the secret from Kaggle and set it as an environment variable
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

print("HF Secrets/API keys have been set")

In [ ]:
def legal_cleaning(text):
    """
    Enhanced cleaning for Indian Law Reports:
    Removes margin ladders (A-H), page headers, and numbers
    while preserving structure for Legal-BERT.
    """

    text = re.sub(r'^\s*[A-H]\s*$', '', text, flags=re.MULTILINE)


    headers = [
        r'\[2020\]\s*1\s*S\.C\.R\.',
        r'SUPREME\s*COURT\s*REPORTS',
        r'CASE\s*LAW\s*REFERENCE',
        r'DIGEST\s*OF\s*CASES'
    ]
    for pattern in headers:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 3. Remove Standalone Page Numbers [cite: 108, 138, 189]
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 4. Standard Cleaning Logic (Preserved from your original code)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r' +', ' ', text)
    text = "".join(char for char in text if char.isprintable() or char in ['\n', '\t'])

    # Final trim to ensure no trailing blank lines from the removed ladders
    text = "\n".join([line.strip() for line in text.split('\n') if line.strip()])

    return text.strip()

In [ ]:
# 1. Base directory for uploaded Kaggle Dataset
DATASET_PATH = "/kaggle/input/datasets/rehanfargose/case-en-pdfs-2020" 

FILE_NAME = "2020_1_1_7_EN.pdf" 

selected_pdf_path = os.path.join(DATASET_PATH, FILE_NAME)

# Verify if the file exists to avoid 'FileNotFound' errors later
if os.path.exists(selected_pdf_path):
    print(f"Successfully located: {selected_pdf_path}")
else:
    print(f"Error: File not found at {selected_pdf_path}")
    print("Available files:", os.listdir(DATASET_PATH))

In [ ]:
def process_document(file_path):
    # 1. Basic validation
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return None

    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}...")

    # 2. Text Extraction using PyMuPDF (fitz)
    raw_text = ""
    try:
        # Open directly from the Kaggle input path
        with fitz.open(file_path) as doc:
            raw_text = "\n".join([page.get_text("text") for page in doc])

        # 3. Pre-processing & Cleaning using your custom function
        cleaned_text = legal_cleaning(raw_text)

        # 4. Output to Kaggle's working directory for your project records
        output_dir = "/kaggle/working/Text_Extracts"
        os.makedirs(output_dir, exist_ok=True)
        
        # Save the cleaned text as a .txt file
        output_file = os.path.join(output_dir, f"{os.path.splitext(file_name)[0]}.txt")
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"Success! Cleaned text saved to: {output_file}")
        return cleaned_text

    except Exception as e:
        print(f"Extraction error: {str(e)}")
        return None

In [ ]:
cleaned_data = process_document(selected_pdf_path )


In [ ]:
# Verify the output
if cleaned_data:
    print(f"Character count of cleaned text: {len(cleaned_data)}")

# **1.5. The Segmentation method for extracting Evidence, Facts and IPCs from cleaned Text file**

In [ ]:
!pip install -U google-genai

In [ ]:
import json
from google import genai
from google.genai import types
# from google.colab import userdata
import os


model_name = 'gemini-3-flash-preview'
# Get the secret from Kaggle and set it as an environment variable
user_secrets = UserSecretsClient()
# os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GOOGLE_API_KEY")

api_key = user_secrets.get_secret("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

def extract_legal_details(legal_text):
    prompt = f"""
    You are a Legal Expert in Indian Law. Analyze the following court text and extract:
    1. FACTS: The core background of the dispute.
    2. EVIDENCE: Specific documents, testimonies, or proofs cited.
    3. IPC: Any Indian Penal Code sections or specific Acts/statutes mentioned/applicable in the case.

    Format the output as a clean JSON not a list.

    TEXT:
    {legal_text}
    """

    # Use the new models.generate_content method
    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="medium"),
            response_mime_type="application/json",
        )
    )


    try:
        data = json.loads(response.text)
        return data
    except json.JSONDecodeError:
        print("Failed to parse JSON")
        return None

In [ ]:
generate_models = [m.name for m in client.models.list() 
                   if 'generateContent' in m.supported_actions]

print("Models you can actually use for generation:")
print(generate_models)

In [ ]:
# Get the extracted facts, evidence and basic IPCs from the text file
result = extract_legal_details(cleaned_data)
# result = {'FACTS': "The appellant, M/S. L. R. Brothers Indo Flora Ltd., a 100% Export Oriented Unit (EOU) engaged in producing cut flowers, sold goods in the Domestic Tariff Area (DTA) between 1998-99 and 2000-01 without prior approval from the Development Commissioner and without achieving the required 20% positive net foreign exchange earning as mandated by the EXIM Policy 1997-2002. The unit had imported inputs like 'Live Rose Plants' and fertilizers duty-free under Notification No. 126/94-Cus. The Revenue issued a show-cause notice demanding customs duty on the imported inputs used for DTA sales, arguing that the breach of EXIM Policy conditions nullified the exemption. The appellant contended that since cut flowers are non-excisable and grown on Indian soil, they should not attract customs duty, and further argued that the 2001 amendment to the notification should apply retrospectively to reduce their liability.", 'EVIDENCE': "Exemption Notification No. 126/94-Cus dated 03.06.1994; Amendment Notification No. 56/01-Cus dated 18.05.2001; EXIM Policy 1997-2002; Show Cause Notice dated 16.03.2001 issued by the Additional Commissioner of Central Excise; Letter dated 06.02.2001 from the appellant seeking ex-post facto approval; Scrutiny of import records for capital goods, 'Live Rose Plants', fertilizers, and planting materials; Export records showing flowers worth Rs. 91.92 lakhs against capital goods imports of Rs. 12.11 crores; CBEC Circular No. 31/2001-Cus dated 24.05.2001; Appendix 42 of the Handbook of Procedure.", 'IPC': 'Customs Act, 1962 (Section 12, Section 25, Section 28, Section 28AB, Section 114A); Central Excise Act, 1944 (Section 3); Customs Tariff Act, 1975; Central Excise Tariff Act, 1985; EXIM Policy 1997-2002.'}
print(result)

In [ ]:
# print(f"The processed facts using {model_name} are:")
case_facts = result.get('FACTS')
# case_facts = "M/S. L.R. Brothers Indo Flora Ltd., a 100% Export Oriented Unit (EOU) producing cut flowers, imported capital goods and raw materials (live rose plants, fertilizers) duty-free under Notification No. 126/94-Cus. Between 1998-99 and 2000-01, the appellant made sales in the Domestic Tariff Area (DTA) totaling Rs. 38,40,537 without obtaining the mandatory prior approval from the Development Commissioner and without achieving the required 20% positive net foreign exchange earning as stipulated by the EXIM Policy 1997-2002. The Revenue issued a show cause notice seeking customs duty on the imported inputs used for the DTA sales of non-excisable goods (cut flowers), calculating the duty at a rate equivalent to the customs duty leviable on the finished articles if imported."
print(case_facts)

In [ ]:
# print(f"The processed evidence using {model_name} are:")
case_evidence = result.get('EVIDENCE')
# case_evidence = "Notification No. 126/94-Cus (dated 03.06.1994); Amendment Notification No. 56/01-Cus (dated 18.05.2001); EXIM Policy 1997-2002 (Paragraphs 9.5, 9.9, 9.29); Letter of approval No. 119(1994)EOB/34/94 (dated 04.05.1994); Appellant's letter seeking ex-post facto approval (dated 06.02.2001); Show cause notice issued by Additional Commissioner, Central Excise, Meerut-I (dated 16.03.2001); CBEC Circular No. 31/2001-Cus (dated 24.05.2001); Import documentation for Live Rose Plants, fertilizers, and greenhouse equipment."

print(case_evidence)

In [ ]:
# print(f"The processed IPCs using {model_name} are:")
case_ipc = result.get('IPC')
# case_ipc = "Customs Act, 1962 (Sections 12, 25, 28, 28AB, 114A); Central Excise Act, 1944 (Section 3); Customs Tariff Act, 1975; Central Excise Tariff Act, 1985; EXIM Policy 1997-2002."
print(case_ipc)

# **Load the DeepSeek-R1-Distill-Llama-8B model**

In [ ]:
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

In [ ]:
# 1. Configuration
max_seq_length = 2048
dtype = None # None for auto-detection (Float16 for T4 GPUs)
load_in_4bit = True #  4-bit quantization to save memory

In [ ]:
# 2. Load the Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
adapter_path = "/kaggle/input/datasets/rehanfargose/deepseek-r1-8b-lora-adapters"

In [ ]:
# Load local LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    model_name = adapter_path, 
    is_trainable = False
)

In [ ]:
FastLanguageModel.for_inference(model)

print(f"Successfully loaded local adapters from: {adapter_path}")

# **2.5. Scrutinize the Evidence based on Facts and Arguments**

In [ ]:
scrutiny_prompt = f"""Analyze the following legal case as a Senior Legal Auditor. 
Perform a granular "Requirement vs. Reality" audit.

### CASE MATERIALS
FACTS:
{case_facts}

EVIDENCE:
{case_evidence}

### INSTRUCTIONS:
For every finding, include:
- THE STATUTORY REQUIREMENT
- THE FACTUAL REALITY
- THE QUANTIFIABLE DISCREPANCY

### Response:
<think>
"""


In [ ]:
# Tokenize prompt
inputs = tokenizer([scrutiny_prompt], return_tensors="pt").to("cuda")

In [ ]:
print(f"Input tensor shape: {inputs.input_ids.shape}")

In [ ]:
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,          
        temperature=0.2,         
        top_p=0.95,
        use_cache=True,          
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

In [ ]:
generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]
scrutiny_result = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("### SCRUTINY REPORT ###")
print(scrutiny_result)

# **4. Precedent Retrieval**

In [ ]:
precedent_prompt = f"""Analyze the following legal details as a Senior Legal Researcher. 
Your goal is to identify relevant Indian Case Law precedents (Supreme Court or High Court) that align with these facts.

### CURRENT CASE FACTS:
{case_facts}

### APPLICABLE IPC/STATUTES:
{case_ipc}

### TASK:
1. Identify 2-3 landmark or relevant Indian precedents.
2. Explain the 'Ratio Decidendi' (legal reasoning) of those cases.
3. State how they apply to the current disputed evidence.

### Response:
<think>
"""

In [ ]:
# Tokenize prompt
inputs = tokenizer([precedent_prompt], return_tensors="pt").to("cuda")

In [ ]:
print(f"Input tensor shape: {inputs.input_ids.shape}")

In [ ]:
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=3000,   
        do_sample=True,        
        temperature=0.6,       
        top_p=0.95,
        use_cache=True,
    )

In [ ]:
generated_tokens1 = outputs[0][inputs.input_ids.shape[-1]:]

precedent_result = tokenizer.decode(generated_tokens1, skip_special_tokens=True)

print("### PRECEDENT REPORT ###")
print(precedent_result)

# **5. Verdict/Sentence Prediction**

In [ ]:
verdict_prompt = f"""You are a Justice of the Supreme Court of India. 
Decide the verdict for the case provided below. 
Your verdict must be based ONLY on the provided materials. Do not reference outside cases unless they are in the context.

### CASE MATERIALS
FACTS: {case_facts}
STATUTES: {case_ipc}
SCRUTINY: {scrutiny_result}

### INSTRUCTIONS:
Pronounce your judgment specifically addressing:
1. ON STATUTORY BREACH
2. ON EVIDENTIARY WEIGHT
3. FINAL ORDER

### Response:
<think>
"""

In [ ]:
# Tokenize prompt
inputs = tokenizer([verdict_prompt], return_tensors="pt").to("cuda")

In [ ]:
print(f"Input tensor shape: {inputs.input_ids.shape}")

In [ ]:
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=3000,     
        do_sample=True,          
        temperature=0.6,         
        top_p=0.95,              
        use_cache=True,
        repetition_penalty=1.1,  
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

In [ ]:
generated_tokens2 = outputs[0][inputs.input_ids.shape[-1]:]

verdict_result = tokenizer.decode(generated_tokens2, skip_special_tokens=True)

print("### VERDICT REPORT ###")
print(verdict_result)

# **6. Summarization**

In [ ]:
summary_prompt = f"""You are a Legal Journalist. Summarize the following legal case in approximately 150 words. 
Use simple, non-technical language that a non-lawyer can understand. 

### CASE DATA:
- **FACTS**: {case_facts}
- **SCRUTINIZED EVIDENCE**: {scrutiny_result}
- **RELEVANT PRECEDENTS**: {precedent_result}
- **PREDICTED VERDICT**: {verdict_result}

### STRUCTURE:
1. The Core Dispute (What happened?)
2. Key Findings (What did the analysis reveal?)
3. Final Verdict & Reasoning (What was decided and why?)

### Response:
<think>
"""

In [ ]:
# Tokenize prompt
inputs = tokenizer([summary_prompt], return_tensors="pt").to("cuda")

In [ ]:
print(f"Input tensor shape: {inputs.input_ids.shape}")

In [ ]:
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,     
        do_sample=True,
        temperature=0.4,         
        top_p=0.9,
        use_cache=True,
    )

In [ ]:
generated_tokens3 = outputs[0][inputs.input_ids.shape[-1]:]

summary_result = tokenizer.decode(generated_tokens3, skip_special_tokens=True)

print("### SUMMARY REPORT ###")
print(summary_result)